# EDA Report: Garbage Classification Dataset

This notebook provides a standalone exploratory data analysis (EDA) for the Kaggle dataset `asdasdasasdas/garbage-classification`. It focuses on class distribution, image geometry, file size patterns, sampled visual inspection, and summary findings for modelling decisions.

In [ ]:
%matplotlib inline

from pathlib import Path
from collections import Counter
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid", context="notebook")
random.seed(42)
np.random.seed(42)

project_root = Path.cwd().expanduser()
dataset_root = project_root / "dataset" / "Garbage classification" / "Garbage classification"
if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset root not found: {dataset_root}")

print(f"Dataset root: {dataset_root}")

In [ ]:
records = []
for category_dir in sorted([p for p in dataset_root.iterdir() if p.is_dir()]):
    for img_path in category_dir.glob('**/*'):
        if img_path.is_file() and img_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
            try:
                with Image.open(img_path) as img:
                    w, h = img.size
                records.append({
                    "category": category_dir.name,
                    "image_path": str(img_path),
                    "width_px": w,
                    "height_px": h,
                    "aspect_ratio": w / h,
                    "file_size_kb": img_path.stat().st_size / 1024.0,
                })
            except Exception:
                continue

eda_df = pd.DataFrame(records)
print("EDA DataFrame shape:", eda_df.shape)
eda_df.head()

In [ ]:
class_counts = eda_df['category'].value_counts().sort_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Class counts
axes[0].bar(class_counts.index, class_counts.values, color="#2E86AB")
axes[0].set_title("Class Distribution")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Image Count")
axes[0].tick_params(axis="x", rotation=25)

# File size
sns.boxplot(data=eda_df, x="category", y="file_size_kb", ax=axes[1], color="#7FB3D5")
axes[1].set_title("File Size Distribution (KB)")
axes[1].tick_params(axis="x", rotation=25)

# Aspect ratio
sns.boxplot(data=eda_df, x="category", y="aspect_ratio", ax=axes[2], color="#A9DFBF")
axes[2].set_title("Aspect Ratio Distribution")
axes[2].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

In [ ]:
summary = eda_df.groupby('category').agg(
    count=('image_path', 'count'),
    mean_width=('width_px', 'mean'),
    mean_height=('height_px', 'mean'),
    mean_size_kb=('file_size_kb', 'mean'),
    mean_aspect=('aspect_ratio', 'mean'),
).round(3)

print("Per-category summary statistics:")
display(summary)

## Key Findings

1. The `trash` category is substantially underrepresented relative to other categories.
2. Image dimensions are uniform (512 x 384), which simplifies preprocessing.
3. A plausible ambiguity pair is `cardboard` vs `paper`, based on similar visual appearance and close channel-intensity tendencies in sampled EDA statistics.
4. This ambiguity claim should be validated quantitatively using confusion-matrix and per-class evaluation on the full test split.
5. For full modelling context and HTML export workflow, refer to `Forum06-garbage_classification_question.ipynb`.
